In [ ]:
# ================================================================
# END-TO-END LLM DATASET PREPARATION AND CURATION PIPELINE
# ================================================================
# Concepts covered:
# Raw Data Collection
# Data Cleaning
# Text Normalisation
# Quality Filtering
# Deduplication
# LLM Tokenisation
# Token-Length Filtering
# Data Structuring
# Train/Validation Split
# Instruction-Tuning Data
# Preference / RLHF-Style Data
# Domain-Specific Data Preparation
# Dataset Storage Formats
# Dataset Statistics
# ================================================================


# ------------------------------------------------
# STEP 1: INSTALL REQUIRED LIBRARIES
# ------------------------------------------------

!pip install pandas datasets transformers -q


# ------------------------------------------------
# STEP 2: IMPORT LIBRARIES
# ------------------------------------------------

import pandas as pd
import re

from datasets import Dataset
from transformers import AutoTokenizer

In [ ]:
# ------------------------------------------------
# STEP 3: COLLECT RAW TEXT DATA
# ------------------------------------------------
# We intentionally include:
# - extra spaces
# - HTML tags
# - duplicates
# - short text
# - empty text
# - special characters
# - long text
# ------------------------------------------------

raw_data = [
    "  Machine learning enables computers to learn from data!!  ",

    "<p>Deep learning is a subset of machine learning.</p>",

    "Deep learning is a subset of machine learning.",

    "ok",

    "Large language models use tokenization    techniques.",

    "Artificial intelligence is transforming industries.",

    "",

    "Transformers are widely used in natural language processing.",

    "Machine learning enables computers to learn from data!!",

    "AI!!! $$$ ###",

    "Python is widely used for machine learning.",

    "This is a very very very very very very very very "
    "very very very very very long sentence that can "
    "be used to demonstrate length based filtering."
]

raw_df = pd.DataFrame({
    "raw_text": raw_data
})

print("=" * 70)
print("STEP 1: RAW DATA")
print("=" * 70)

print(raw_df)


STEP 1: RAW DATA
                                             raw_text
0     Machine learning enables computers to learn ...
1   <p>Deep learning is a subset of machine learni...
2      Deep learning is a subset of machine learning.
3                                                  ok
4   Large language models use tokenization    tech...
5   Artificial intelligence is transforming indust...
6                                                    
7   Transformers are widely used in natural langua...
8   Machine learning enables computers to learn fr...
9                                       AI!!! $$$ ###
10        Python is widely used for machine learning.
11  This is a very very very very very very very v...


In [ ]:
# ------------------------------------------------
# STEP 4: DATA CLEANING
# ------------------------------------------------
# Remove:
# - HTML tags
# - excessive whitespace
# - unwanted special characters
# - leading/trailing spaces
# ------------------------------------------------

def clean_text(text):

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove unwanted special characters
    text = re.sub(r"[^\w\s.,!?'-]", "", text)

    # Replace multiple spaces with one
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


raw_df["cleaned_text"] = raw_df["raw_text"].apply(clean_text)

print("\n" + "=" * 70)
print("STEP 2: CLEANED DATA")
print("=" * 70)

print(raw_df[["cleaned_text"]])


STEP 2: CLEANED DATA
                                         cleaned_text
0   Machine learning enables computers to learn fr...
1      Deep learning is a subset of machine learning.
2      Deep learning is a subset of machine learning.
3                                                  ok
4   Large language models use tokenization techniq...
5   Artificial intelligence is transforming indust...
6                                                    
7   Transformers are widely used in natural langua...
8   Machine learning enables computers to learn fr...
9                                               AI!!!
10        Python is widely used for machine learning.
11  This is a very very very very very very very v...


In [ ]:
# ------------------------------------------------
# STEP 5: TEXT NORMALISATION
# ------------------------------------------------
# Normalisation includes:
# - converting text to lowercase
# - standardising whitespace
# ------------------------------------------------

def normalize_text(text):

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text


raw_df["normalized_text"] = (
    raw_df["cleaned_text"]
    .apply(normalize_text)
)

print("\n" + "=" * 70)
print("STEP 3: NORMALISED DATA")
print("=" * 70)

print(raw_df[[
    "normalized_text"
]])



STEP 3: NORMALISED DATA
                                      normalized_text
0   machine learning enables computers to learn fr...
1      deep learning is a subset of machine learning.
2      deep learning is a subset of machine learning.
3                                                  ok
4   large language models use tokenization techniq...
5   artificial intelligence is transforming indust...
6                                                    
7   transformers are widely used in natural langua...
8   machine learning enables computers to learn fr...
9                                               ai!!!
10        python is widely used for machine learning.
11  this is a very very very very very very very v...


In [7]:
# ------------------------------------------------
# STEP 6: QUALITY FILTERING
# ------------------------------------------------
# Remove:
# - empty records
# - very short records
# - extremely long records
# - records containing insufficient alphabetic content
# ------------------------------------------------

def quality_check(text):

    words = text.split()

    # Empty text
    if len(text.strip()) == 0:
        return False

    # Minimum number of words
    if len(words) < 4:
        return False

    # Maximum number of words
    if len(words) > 30:
        return False

    # Minimum alphabetic characters
    alphabetic_chars = sum(
        character.isalpha()
        for character in text
    )

    if alphabetic_chars < 10:
        return False

    return True


raw_df["quality_pass"] = (
    raw_df["normalized_text"]
    .apply(quality_check)
)

quality_df = raw_df[
    raw_df["quality_pass"]
].copy()

print("\n" + "=" * 70)
print("STEP 4: QUALITY FILTERING")
print("=" * 70)

print("Records before filtering :", len(raw_df))
print("Records after filtering  :", len(quality_df))

print("\nQuality-passed records:")
print(quality_df[["normalized_text"]])


# ------------------------------------------------
# STEP 7: DUPLICATE REMOVAL
# ------------------------------------------------

before_dedup = len(quality_df)

quality_df = quality_df.drop_duplicates(
    subset=["normalized_text"]
).reset_index(drop=True)

after_dedup = len(quality_df)

print("\n" + "=" * 70)
print("STEP 5: DEDUPLICATION")
print("=" * 70)

print("Records before deduplication :", before_dedup)
print("Records after deduplication  :", after_dedup)

print("\nDeduplicated dataset:")
print(quality_df[["normalized_text"]])


STEP 4: QUALITY FILTERING
Records before filtering : 12
Records after filtering  : 9

Quality-passed records:
                                      normalized_text
0   machine learning enables computers to learn fr...
1      deep learning is a subset of machine learning.
2      deep learning is a subset of machine learning.
4   large language models use tokenization techniq...
5   artificial intelligence is transforming indust...
7   transformers are widely used in natural langua...
8   machine learning enables computers to learn fr...
10        python is widely used for machine learning.
11  this is a very very very very very very very v...

STEP 5: DEDUPLICATION
Records before deduplication : 9
Records after deduplication  : 7

Deduplicated dataset:
                                     normalized_text
0  machine learning enables computers to learn fr...
1     deep learning is a subset of machine learning.
2  large language models use tokenization techniq...
3  artificial intelligenc

In [8]:
# ------------------------------------------------
# STEP 8: LOAD A REAL HUGGING FACE TOKENIZER
# ------------------------------------------------
# Instead of simple word splitting, we use a
# subword tokenizer used with transformer models.
# ------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)


# ------------------------------------------------
# STEP 9: TOKENISATION
# ------------------------------------------------

def tokenize_text(text):

    return tokenizer.tokenize(text)


def convert_to_token_ids(text):

    return tokenizer.encode(
        text,
        add_special_tokens=True
    )


quality_df["tokens"] = (
    quality_df["normalized_text"]
    .apply(tokenize_text)
)

quality_df["token_ids"] = (
    quality_df["normalized_text"]
    .apply(convert_to_token_ids)
)

quality_df["token_count"] = (
    quality_df["token_ids"]
    .apply(len)
)

print("\n" + "=" * 70)
print("STEP 6: LLM TOKENISATION")
print("=" * 70)

for _, row in quality_df.iterrows():

    print("\nText:")
    print(row["normalized_text"])

    print("Tokens:")
    print(row["tokens"])

    print("Token IDs:")
    print(row["token_ids"])

    print("Token Count:")
    print(row["token_count"])



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


STEP 6: LLM TOKENISATION

Text:
machine learning enables computers to learn from data!!
Tokens:
['machine', 'learning', 'enables', 'computers', 'to', 'learn', 'from', 'data', '!', '!']
Token IDs:
[101, 3698, 4083, 12939, 7588, 2000, 4553, 2013, 2951, 999, 999, 102]
Token Count:
12

Text:
deep learning is a subset of machine learning.
Tokens:
['deep', 'learning', 'is', 'a', 'subset', 'of', 'machine', 'learning', '.']
Token IDs:
[101, 2784, 4083, 2003, 1037, 16745, 1997, 3698, 4083, 1012, 102]
Token Count:
11

Text:
large language models use tokenization techniques.
Tokens:
['large', 'language', 'models', 'use', 'token', '##ization', 'techniques', '.']
Token IDs:
[101, 2312, 2653, 4275, 2224, 19204, 3989, 5461, 1012, 102]
Token Count:
10

Text:
artificial intelligence is transforming industries.
Tokens:
['artificial', 'intelligence', 'is', 'transforming', 'industries', '.']
Token IDs:
[101, 7976, 4454, 2003, 17903, 6088, 1012, 102]
Token Count:
8

Text:
transformers are widely used in n

In [9]:
# ------------------------------------------------
# STEP 10: TOKEN-LENGTH FILTERING
# ------------------------------------------------
# LLM datasets are often filtered according to
# token length rather than only word count.
# ------------------------------------------------

MAX_TOKENS = 20

token_filtered_df = quality_df[
    quality_df["token_count"] <= MAX_TOKENS
].reset_index(drop=True)

print("\n" + "=" * 70)
print("STEP 7: TOKEN-LENGTH FILTERING")
print("=" * 70)

print("Maximum allowed tokens:", MAX_TOKENS)

print("\nFinal text records:")
print(
    token_filtered_df[
        ["normalized_text", "token_count"]
    ]
)



STEP 7: TOKEN-LENGTH FILTERING
Maximum allowed tokens: 20

Final text records:
                                     normalized_text  token_count
0  machine learning enables computers to learn fr...           12
1     deep learning is a subset of machine learning.           11
2  large language models use tokenization techniq...           10
3  artificial intelligence is transforming indust...            8
4  transformers are widely used in natural langua...           11
5        python is widely used for machine learning.           10


In [10]:
# ------------------------------------------------
# STEP 11: STRUCTURE THE DATASET
# ------------------------------------------------

structured_df = token_filtered_df[
    ["normalized_text", "token_count"]
].copy()

structured_df.columns = [
    "text",
    "token_count"
]

structured_df.insert(
    0,
    "id",
    range(1, len(structured_df) + 1)
)

print("\n" + "=" * 70)
print("STEP 8: STRUCTURED DATASET")
print("=" * 70)

print(structured_df)


# ------------------------------------------------
# STEP 12: CREATE TRAIN / VALIDATION SPLIT
# ------------------------------------------------

text_dataset = Dataset.from_pandas(
    structured_df
)

split_dataset = text_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

print("\n" + "=" * 70)
print("STEP 9: TRAIN / VALIDATION SPLIT")
print("=" * 70)

print(split_dataset)

print("\nTraining examples:")
print(split_dataset["train"])

print("\nValidation examples:")
print(split_dataset["test"])


STEP 8: STRUCTURED DATASET
   id                                               text  token_count
0   1  machine learning enables computers to learn fr...           12
1   2     deep learning is a subset of machine learning.           11
2   3  large language models use tokenization techniq...           10
3   4  artificial intelligence is transforming indust...            8
4   5  transformers are widely used in natural langua...           11
5   6        python is widely used for machine learning.           10

STEP 9: TRAIN / VALIDATION SPLIT
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'token_count'],
        num_rows: 4
    })
    test: Dataset({
        features: ['id', 'text', 'token_count'],
        num_rows: 2
    })
})

Training examples:
Dataset({
    features: ['id', 'text', 'token_count'],
    num_rows: 4
})

Validation examples:
Dataset({
    features: ['id', 'text', 'token_count'],
    num_rows: 2
})


In [11]:
# ------------------------------------------------
# STEP 13: CREATE INSTRUCTION-TUNING DATA
# ------------------------------------------------
# These are meaningful prompt-response examples.
# ------------------------------------------------

instruction_data = [

    {
        "instruction": "Explain machine learning.",
        "input": "",
        "output":
        "Machine learning is a method that enables "
        "computers to learn patterns from data."
    },

    {
        "instruction": "What is deep learning?",
        "input": "",
        "output":
        "Deep learning is a branch of machine learning "
        "that uses neural networks with multiple layers."
    },

    {
        "instruction": "What are transformers used for?",
        "input": "",
        "output":
        "Transformers are neural network architectures "
        "widely used for natural language processing "
        "and other AI tasks."
    },

    {
        "instruction": "Why is tokenization important?",
        "input": "",
        "output":
        "Tokenization converts text into smaller units "
        "that can be represented as numerical token IDs "
        "for a language model."
    },

    {
        "instruction":
        "Name a programming language commonly used "
        "in machine learning.",
        "input": "",
        "output":
        "Python is widely used for machine learning."
    }
]

instruction_df = pd.DataFrame(
    instruction_data
)

print("\n" + "=" * 70)
print("STEP 10: INSTRUCTION-TUNING DATA")
print("=" * 70)

print(instruction_df)



STEP 10: INSTRUCTION-TUNING DATA
                                         instruction input  \
0                          Explain machine learning.         
1                             What is deep learning?         
2                    What are transformers used for?         
3                     Why is tokenization important?         
4  Name a programming language commonly used in m...         

                                              output  
0  Machine learning is a method that enables comp...  
1  Deep learning is a branch of machine learning ...  
2  Transformers are neural network architectures ...  
3  Tokenization converts text into smaller units ...  
4        Python is widely used for machine learning.  


In [12]:
# ------------------------------------------------
# STEP 14: CONVERT INSTRUCTION DATA TO
# HUGGING FACE DATASET
# ------------------------------------------------

instruction_dataset = Dataset.from_pandas(
    instruction_df
)

print("\n" + "=" * 70)
print("STEP 11: HUGGING FACE INSTRUCTION DATASET")
print("=" * 70)

print(instruction_dataset)

print("\nFirst example:")
print(instruction_dataset[0])



STEP 11: HUGGING FACE INSTRUCTION DATASET
Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

First example:
{'instruction': 'Explain machine learning.', 'input': '', 'output': 'Machine learning is a method that enables computers to learn patterns from data.'}


In [13]:
# ------------------------------------------------
# STEP 15: CREATE PREFERENCE / RLHF-STYLE DATA
# ------------------------------------------------
# chosen   = preferred response
# rejected = less preferred response
# ------------------------------------------------

preference_data = [

    {
        "prompt": "Explain machine learning.",

        "chosen":
        "Machine learning enables computers to "
        "learn patterns from data.",

        "rejected":
        "Machine learning is a computer program."
    },

    {
        "prompt": "What is tokenization?",

        "chosen":
        "Tokenization converts text into smaller "
        "units called tokens.",

        "rejected":
        "Tokenization means converting computers "
        "into tokens."
    },

    {
        "prompt": "What is deep learning?",

        "chosen":
        "Deep learning uses neural networks with "
        "multiple layers to learn representations "
        "from data.",

        "rejected":
        "Deep learning means learning something deeply."
    }
]

preference_df = pd.DataFrame(
    preference_data
)

print("\n" + "=" * 70)
print("STEP 12: PREFERENCE / RLHF-STYLE DATA")
print("=" * 70)

print(preference_df)


STEP 12: PREFERENCE / RLHF-STYLE DATA
                      prompt  \
0  Explain machine learning.   
1      What is tokenization?   
2     What is deep learning?   

                                              chosen  \
0  Machine learning enables computers to learn pa...   
1  Tokenization converts text into smaller units ...   
2  Deep learning uses neural networks with multip...   

                                            rejected  
0            Machine learning is a computer program.  
1  Tokenization means converting computers into t...  
2     Deep learning means learning something deeply.  


In [14]:

# ------------------------------------------------
# STEP 16: HUGGING FACE PREFERENCE DATASET
# ------------------------------------------------

preference_dataset = Dataset.from_pandas(
    preference_df
)

print("\nHugging Face Preference Dataset:")
print(preference_dataset)



Hugging Face Preference Dataset:
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 3
})


In [15]:
# ------------------------------------------------
# STEP 17: DOMAIN-SPECIFIC DATA PREPARATION
# ------------------------------------------------
# Example domain: EDUCATION
# ------------------------------------------------

domain_data = [

    {
        "domain": "education",

        "instruction":
        "What is supervised learning?",

        "response":
        "Supervised learning trains a model "
        "using labelled examples."
    },

    {
        "domain": "education",

        "instruction":
        "What is a neural network?",

        "response":
        "A neural network is a computational model "
        "inspired by interconnected neurons."
    },

    {
        "domain": "education",

        "instruction":
        "What is overfitting?",

        "response":
        "Overfitting occurs when a model learns the "
        "training data too closely and performs poorly "
        "on unseen data."
    },

    {
        "domain": "finance",

        "instruction":
        "What is inflation?",

        "response":
        "Inflation is a general increase in the prices "
        "of goods and services over time."
    }
]

domain_df = pd.DataFrame(domain_data)

print("\n" + "=" * 70)
print("STEP 13: DOMAIN-SPECIFIC DATA")
print("=" * 70)

print(domain_df)


# ------------------------------------------------
# STEP 18: FILTER A PARTICULAR DOMAIN
# ------------------------------------------------

education_df = domain_df[
    domain_df["domain"] == "education"
].reset_index(drop=True)

print("\nEducation Domain Dataset:")
print(education_df)



STEP 13: DOMAIN-SPECIFIC DATA
      domain                   instruction  \
0  education  What is supervised learning?   
1  education     What is a neural network?   
2  education          What is overfitting?   
3    finance            What is inflation?   

                                            response  
0  Supervised learning trains a model using label...  
1  A neural network is a computational model insp...  
2  Overfitting occurs when a model learns the tra...  
3  Inflation is a general increase in the prices ...  

Education Domain Dataset:
      domain                   instruction  \
0  education  What is supervised learning?   
1  education     What is a neural network?   
2  education          What is overfitting?   

                                            response  
0  Supervised learning trains a model using label...  
1  A neural network is a computational model insp...  
2  Overfitting occurs when a model learns the tra...  


In [16]:
# ------------------------------------------------
# STEP 19: SAVE DATASET IN CSV FORMAT
# ------------------------------------------------

instruction_df.to_csv(
    "instruction_dataset.csv",
    index=False
)

print("\nCSV file saved.")


CSV file saved.


In [17]:
# ------------------------------------------------
# STEP 20: SAVE DATASET IN JSON FORMAT
# ------------------------------------------------

instruction_df.to_json(
    "instruction_dataset.json",
    orient="records",
    indent=2
)

print("JSON file saved.")

JSON file saved.


In [18]:
# ------------------------------------------------
# STEP 21: SAVE DATASET IN JSONL FORMAT
# ------------------------------------------------
# JSONL is commonly used for LLM datasets because
# each line represents one independent example.
# ------------------------------------------------

instruction_df.to_json(
    "instruction_dataset.jsonl",
    orient="records",
    lines=True
)

print("JSONL file saved.")

JSONL file saved.


In [19]:
# ------------------------------------------------
# STEP 22: SAVE HUGGING FACE DATASET
# ------------------------------------------------

instruction_dataset.save_to_disk(
    "llm_instruction_dataset"
)

print("Hugging Face Dataset saved.")


Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Hugging Face Dataset saved.


In [21]:
# ------------------------------------------------
# STEP 23: DATASET STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 14: FINAL DATASET STATISTICS")
print("=" * 70)

print(
    "Raw records:",
    len(raw_df)
)

print(
    "After quality filtering:",
    len(quality_df)
)

print(
    "Final text records:",
    len(structured_df)
)

print(
    "Instruction records:",
    len(instruction_df)
)

print(
    "Preference records:",
    len(preference_df)
)

print(
    "Domain-specific records:",
    len(domain_df)
)

print(
    "Average token count:",
    round(
        structured_df["token_count"].mean(),
        2
    )
)

print(
    "Maximum token count:",
    structured_df["token_count"].max()
)

print(
    "Minimum token count:",
    structured_df["token_count"].min()
)


STEP 14: FINAL DATASET STATISTICS
Raw records: 12
After quality filtering: 7
Final text records: 6
Instruction records: 5
Preference records: 3
Domain-specific records: 4
Average token count: 10.33
Maximum token count: 12
Minimum token count: 8
